In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lag
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor, LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from sklearn.feature_selection import mutual_info_regression

DATA_DIR = "../data"
OUTPUT_MERGED = os.path.join(DATA_DIR, "merged_gold_data.csv")


def load_and_prepare_csv(filename):
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    if 'date' in df.columns:
        df.rename(columns={'date': 'Date'}, inplace=True)
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
    return df

def remove_outliers_iqr(data, column):  
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] >= lower_bound) & (data[column] <= upper_bound)]

def process_outliers_for_files(file_paths):
    for file_path in file_paths:
        print(f"Xử lý file: {os.path.basename(file_path)}")
        df = pd.read_csv(file_path)
        numeric_columns = df.select_dtypes(include=["float64", "int64"]).columns
        df_no_outliers = df.copy()
        print(f"Số dòng ban đầu: {df.shape[0]}")
        for col in numeric_columns:
            before = df_no_outliers.shape[0]
            df_no_outliers = remove_outliers_iqr(df_no_outliers, col)
            after = df_no_outliers.shape[0]
            print(f"Đã loại bỏ ngoại lai cột '{col}': {before - after} dòng (còn lại: {after})")

        for col in numeric_columns:
            print(f"Vẽ boxplot cho cột: {col} trong file {os.path.basename(file_path)}")
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            sns.boxplot(x=df[col], ax=axes[0], color="salmon")
            axes[0].set_title(f"{col} – Trước khi loại bỏ ngoại lai")
            sns.boxplot(x=df_no_outliers[col], ax=axes[1], color="lightgreen")
            axes[1].set_title(f"{col} – Sau khi loại bỏ ngoại lai")
            plt.suptitle(f"Boxplot – {col} ({os.path.basename(file_path)})", fontsize=14)
            plt.tight_layout()
            plt.show()

        cleaned_path = os.path.join(DATA_DIR, os.path.basename(file_path).replace(".csv", "_cleaned.csv"))
        df_no_outliers.to_csv(cleaned_path, index=False)
        print(f"Đã lưu file sạch: {cleaned_path}")

def merge_cleaned_files():
    gold_df = load_and_prepare_csv("gold_cleaned.csv")
    oil_df = load_and_prepare_csv("oil_cleaned.csv")
    dxy_df = load_and_prepare_csv("dxy_cleaned.csv")
    sp500_df = load_and_prepare_csv("sp500_cleaned.csv")

    merged_df = gold_df.merge(oil_df, on="Date", how="outer")
    merged_df = merged_df.merge(dxy_df, on="Date", how="outer")
    merged_df = merged_df.merge(sp500_df, on="Date", how="outer")

    merged_df = (
        merged_df.sort_values("Date")
        .drop_duplicates(subset=["Date"])
        .reset_index(drop=True)
        .ffill()
    )

    merged_df.to_csv(OUTPUT_MERGED, index=False)
    print(f"Đã merge và lưu file: {OUTPUT_MERGED}")

def evaluate_feature_correlation(df):
    features = df.select_dtypes(include=["number"]).drop(columns=["gold_last"]).copy()
    target = df["gold_last"]

    mi_scores = mutual_info_regression(features, target, random_state=42)
    mi_df = pd.DataFrame({"Feature": features.columns, "MI Score": mi_scores})
    mi_df = mi_df.sort_values("MI Score", ascending=False)

    print("\n Mutual Information giữa đặc trưng và gold_last:")
    print(mi_df.to_string(index=False))

    plt.figure(figsize=(10, 6))
    sns.barplot(x="MI Score", y="Feature", data=mi_df, palette="viridis")
    plt.title("Mutual Information Score với gold_last")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 10))
    corr = df.select_dtypes(include=["number"]).corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Ma trận tương quan Pearson")
    plt.show()

def train_model(n_lags=5):
    spark = SparkSession.builder.appName("GoldPricePrediction").getOrCreate()
    
    df_pd = pd.read_csv(OUTPUT_MERGED)
    df_pd = df_pd.dropna()
    df_pd["Date"] = pd.to_datetime(df_pd["Date"])
    df_pd = df_pd.sort_values("Date")

    df = spark.createDataFrame(df_pd)

    base_features = [c for c in df.columns if c.startswith("dxy_") or c.startswith("sp500_") or c.startswith("gold_") or c.startswith("oil_")]
    base_features = [f for f in base_features if f not in ["gold_last", "gold_change_percent"]]

    window_spec = Window.orderBy("Date")
    for feat in base_features:
        for i in range(1, n_lags + 1):
            df = df.withColumn(f"{feat}_lag_{i}", lag(col(feat), i).over(window_spec))

    df = df.withColumn("target", lag("gold_last", -1).over(window_spec))
    df = df.dropna()

    # df_pd = df.toPandas()
    # for col_name in df_pd.select_dtypes(include="number").columns:
    #     Q1 = df_pd[col_name].quantile(0.25)
    #     Q3 = df_pd[col_name].quantile(0.75)
    #     IQR = Q3 - Q1
    #     lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    #     df_pd = df_pd[(df_pd[col_name] >= lower) & (df_pd[col_name] <= upper)]

    numeric_cols = df_pd.select_dtypes(include="number").columns.drop("target")
    df_pd[numeric_cols] = (df_pd[numeric_cols] - df_pd[numeric_cols].mean()) / df_pd[numeric_cols].std()

    df = spark.createDataFrame(df_pd)
    train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

    drop_cols = ["Date", "gold_last", "gold_change_percent", "target"]
    feature_cols = [c for c in df.columns if c not in drop_cols]

    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
    train_vec = assembler.transform(train_df)
    test_vec = assembler.transform(test_df)

    scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
    scaler_model = scaler.fit(train_vec)
    train_scaled = scaler_model.transform(train_vec)
    test_scaled = scaler_model.transform(test_vec)

    models = {
        "Random Forest": RandomForestRegressor(labelCol="target", featuresCol="features", numTrees=100, seed=42),
        "Linear Regression": LinearRegression(labelCol="target", featuresCol="features")
    }

    results = []
    evaluator = RegressionEvaluator(labelCol="target", predictionCol="prediction")

    for name, model in models.items():
        fitted_model = model.fit(train_scaled)
        train_pred = fitted_model.transform(train_scaled)
        test_pred = fitted_model.transform(test_scaled)

        mae = evaluator.setMetricName("mae").evaluate(test_pred)
        mse = evaluator.setMetricName("mse").evaluate(test_pred)
        rmse = evaluator.setMetricName("rmse").evaluate(test_pred)
        r2 = evaluator.setMetricName("r2").evaluate(test_pred)
        r2_train = evaluator.evaluate(train_pred)
        mean_target = test_pred.select("target").rdd.map(lambda x: x[0]).mean()
        accuracy = 1 - (mae / mean_target)

        print(f"\n🔎 Mô hình: {name}")
        print(f"MAE  : {mae:.4f}")
        print(f"MSE  : {mse:.4f}")
        print(f"RMSE : {rmse:.4f}")
        print(f"R2 Test : {r2:.4f}")
        print(f"R2 Train: {r2_train:.4f}")
        print(f"Accuracy: {accuracy:.4f}")

        overfit_status = "Tốt"
        if r2_train - r2 > 0.1:
            overfit_status = "Overfit"
        elif r2 - r2_train > 0.1:
            overfit_status = "Underfit"

        results.append({
            "Model": name,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2 Test": r2,
            "R2 Train": r2_train,
            "Accuracy": accuracy,
            "Fit Status": overfit_status
        })

    df_result = pd.DataFrame(results).sort_values("R2 Test", ascending=False)
    print("\n Bảng so sánh mô hình:")
    print(df_result.to_string(index=False))


In [ ]:
file_paths = [
    os.path.join(DATA_DIR, "gold_cleaned.csv"),
    os.path.join(DATA_DIR, "oil_cleaned.csv"),
    os.path.join(DATA_DIR, "sp500_cleaned.csv"),
    os.path.join(DATA_DIR, "dxy_cleaned.csv"),
]

process_outliers_for_files(file_paths)

In [ ]:
merge_cleaned_files()


In [ ]:
merged_df = pd.read_csv(OUTPUT_MERGED)
evaluate_feature_correlation(merged_df)

In [ ]:
train_model()
